<div style="padding: 1em 0.5em; color: #fff; background-color: #0969da; font-size: 1.2em;">
    Lab 3c — Étudiant hybride k-mer + CNN avec distillation
</div>
<div style="border-left: 2px solid #0969da; min-height: 1.5em;margin-left: 1em;padding: 1em;">
    - Combiner les caractéristiques k-mer artisanales avec les motifs CNN appris via fusion tardive<br>
    - Améliorer le modèle hybride avec la distillation de connaissances à partir du professeur Evo2<br>
    - Analyser les poids de fusion appris et comparer avec les modèles de référence<br>
</div>

#### **Votre identité**

Double-cliquez sur cette cellule et complétez, puis exécutez-la (`Maj + Entrée`).

- **Nom & prénom :** _à compléter_
- **Groupe / binôme :** _à compléter_
- **Date :** _à compléter_

<div style="height: 3px; margin: 2em 0 1.5em 0; background: linear-gradient(90deg, #0969da 0%, #0969da 55%, rgba(9,105,218,0.15) 100%);"></div>

#### **Introduction**

Le student du notebook précédent (k-mer + MLP) a atteint ses limites de représentation.
Ici, nous combinons deux représentations complémentaires :

- **k-mers** : caractéristiques artisanales, robustes mais perdent l'ordre des nucléotides
- **CNN one-hot** : apprend des motifs locaux et positionnels directement depuis la séquence

La **fusion tardive** combine ces deux flux après leur encodage, avec des poids appris.
La distillation guide cette fusion avec les cibles douces du teacher Evo2.

**Configuration optimale (basée sur les tests)** :
- Temperature = 2.0
- Alpha = 0.3
- Dropout = 0.3

**Attendu :** ~0.84 de précision (meilleur que le student k-mer seul à ~0.84)

<img src="https://raw.githubusercontent.com/Bioinformatics-UM6P/AfricaBP-workshop-handson-26/main/lab3/assets/Hybrid%20kmer%20CNN%20Distillation%20Pipeline.png" width="600"/>

In [14]:
import sys
sys.path.append("src")

from pathlib import Path
import torch
import torch.nn as nn

from data import load_all
from embeddings import load_supervised_embeddings
from featurize import kmer_matrix, one_hot_batch
from models.classifier_heads import MLPHead
from models.baselines import OneHotCNN
from models.distillation import train_student
from eval import evaluate_logits, count_params, measure_latency_torch

EMB_DIR = "../../2-data/embeddings"
PROCESSED_DIR = "../../2-data/processed"
TEACHER_PATH = Path("../../2-data/models/teacher_mlp.pt")

In [15]:
# Charger les embeddings et le teacher
X_train_emb, y_train, ids_train = load_supervised_embeddings(EMB_DIR, "train")
X_val_emb, y_val, ids_val = load_supervised_embeddings(EMB_DIR, "val")

ckpt = torch.load(TEACHER_PATH)
teacher = MLPHead(d_in=ckpt["d_in"])
teacher.load_state_dict(ckpt["state_dict"])
teacher.eval()
print("teacher rechargé depuis", TEACHER_PATH)

teacher rechargé depuis ../../2-data/models/teacher_mlp.pt


/tmp/ipykernel_1228747/4162923936.py:5: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(TEACHER_PATH)


<div style="height: 3px; margin: 2em 0 1.5em 0; background: linear-gradient(90deg, #0969da 0%, #0969da 55%, rgba(9,105,218,0.15) 100%);"></div>

#### **Préparer les données : aligner les séquences et calculer les caractéristiques**

In [16]:
# Récupérer les séquences des mêmes fenêtres que les embeddings
splits = load_all(PROCESSED_DIR, max_rows=None)
train_df = splits["train"].set_index("id")
val_df = splits["val"].set_index("id")

train_seqs = train_df.loc[ids_train, "sequence"].tolist()
val_seqs = val_df.loc[ids_val, "sequence"].tolist()

# Caractéristiques k-mer
X_train_kmer = torch.tensor(kmer_matrix(train_seqs, k=4), dtype=torch.float32)
X_val_kmer = torch.tensor(kmer_matrix(val_seqs, k=4), dtype=torch.float32)

# Caractéristiques one-hot pour CNN
WINDOW = 200
X_train_oh = torch.tensor(one_hot_batch(train_seqs, length=WINDOW))
X_val_oh = torch.tensor(one_hot_batch(val_seqs, length=WINDOW))

y_train_t = torch.tensor(y_train, dtype=torch.float32)
y_val_t = torch.tensor(y_val, dtype=torch.float32)

print("k-mer shape:", X_train_kmer.shape)
print("one-hot shape:", X_train_oh.shape)

k-mer shape: torch.Size([4000, 256])
one-hot shape: torch.Size([4000, 4, 200])


In [17]:
# Calculer les cibles douces du teacher
with torch.no_grad():
    teacher_logits_train = teacher(torch.tensor(X_train_emb, dtype=torch.float32))
    teacher_val_logits = teacher(torch.tensor(X_val_emb, dtype=torch.float32)).numpy()

teacher_metrics = evaluate_logits(teacher_val_logits, y_val)
print("teacher metrics:", teacher_metrics)

teacher metrics: {'accuracy': 0.959, 'f1': 0.9595261599210266}


<div style="height: 3px; margin: 2em 0 1.5em 0; background: linear-gradient(90deg, #0969da 0%, #0969da 55%, rgba(9,105,218,0.15) 100%);"></div>

#### **Définir le modèle hybride avec fusion tardive**

In [18]:
class HybridKmerCNN(nn.Module):
    """Modèle hybride : fusion tardive de k-mers et CNN one-hot."""
    
    def __init__(self, kmer_dim, seq_len, fusion_dim=128, dropout=0.3, cnn_channels=32):
        super().__init__()
        
        # Branche k-mer -> MLP
        self.kmer_mlp = nn.Sequential(
            nn.Linear(kmer_dim, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, fusion_dim)
        )
        
        # Branche CNN one-hot (extraire les features avant le head)
        self.cnn = OneHotCNN(seq_len=seq_len, channels=cnn_channels)
        self.cnn_proj = nn.Sequential(
            nn.Linear(cnn_channels, fusion_dim),  # CNN features = channels (32)
            nn.ReLU(),
            nn.Dropout(dropout)
        )
        
        # Fusion avec poids appris
        self.fusion_weight = nn.Parameter(torch.tensor([0.5, 0.5]))
        
        # Classification finale
        self.classifier = nn.Sequential(
            nn.Linear(fusion_dim, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 1)
        )
    
    def forward(self, x_kmer, x_oh):
        # Encoder les deux branches
        kmer_feat = self.kmer_mlp(x_kmer)
        
        # Extraire les features du CNN avant le head final
        cnn_feats = self.cnn.net(x_oh).squeeze(-1)  # (B, channels)
        cnn_feat = self.cnn_proj(cnn_feats)
        
        # Fusion pondérée avec softmax pour normaliser les poids
        w = torch.softmax(self.fusion_weight, dim=0)
        fused = w[0] * kmer_feat + w[1] * cnn_feat
        
        # Classification
        logits = self.classifier(fused)
        return logits

<div style="height: 3px; margin: 2em 0 1.5em 0; background: linear-gradient(90deg, #0969da 0%, #0969da 55%, rgba(9,105,218,0.15) 100%);"></div>

#### **Entraîner le student hybride avec distillation**

In [19]:
# Configuration d'entraînement (basée sur les meilleurs résultats des tests)
N_EPOCHS = 100
TEMPERATURE = 2.0
ALPHA = 0.3
LR = 1e-3
DROPOUT = 0.3

In [20]:
# Instancier le modèle hybride
hybrid = HybridKmerCNN(kmer_dim=X_train_kmer.shape[1], seq_len=WINDOW, dropout=DROPOUT)
optimizer = torch.optim.Adam(hybrid.parameters(), lr=LR)

print(f"Hybrid model parameters: {count_params(hybrid):,}")

Hybrid model parameters: 47,780


In [21]:
# Boucle d'entraînement avec distillation
n = X_train_kmer.shape[0]
batch_size = 256

for epoch in range(N_EPOCHS):
    perm = torch.randperm(n)
    epoch_loss = 0.0
    
    for start in range(0, n, batch_size):
        idx = perm[start:start + batch_size]
        optimizer.zero_grad()
        
        # Forward pass
        student_logits = hybrid(X_train_kmer[idx], X_train_oh[idx])
        
        # Perte hybride : étiquettes dures + cibles douces du teacher
        # Perte hard labels
        loss_hard = torch.nn.functional.binary_cross_entropy_with_logits(
            student_logits.squeeze(), y_train_t[idx]
        )
        
        # Perte soft targets (distillation) - pour classification binaire
        # On empile les logits pour créer une distribution 2D (batch, 2)
        teacher_logits_batch = teacher_logits_train[idx]
        student_logits_batch = student_logits.squeeze()
        
        # Créer des logits 2D: [logit_negatif, logit_positif]
        teacher_logits_2d = torch.stack([-teacher_logits_batch, teacher_logits_batch], dim=1)
        student_logits_2d = torch.stack([-student_logits_batch, student_logits_batch], dim=1)
        
        teacher_soft = torch.nn.functional.softmax(teacher_logits_2d / TEMPERATURE, dim=1)
        student_soft = torch.nn.functional.log_softmax(student_logits_2d / TEMPERATURE, dim=1)
        
        loss_soft = torch.nn.functional.kl_div(
            student_soft, teacher_soft, reduction='batchmean'
        ) * (TEMPERATURE ** 2)
        
        # Perte totale
        loss = ALPHA * loss_hard + (1 - ALPHA) * loss_soft
        
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item() * len(idx)
    
    if (epoch + 1) % 20 == 0 or epoch == 0:
        print(f"epoch {epoch+1}/{N_EPOCHS}: loss={epoch_loss/n:.4f}")

epoch 1/100: loss=2.1308
epoch 20/100: loss=0.8556
epoch 40/100: loss=0.7079
epoch 60/100: loss=0.5711
epoch 80/100: loss=0.3846
epoch 100/100: loss=0.2391


#### **Évaluer le student hybride**

In [22]:
hybrid.eval()
with torch.no_grad():
    val_logits = hybrid(X_val_kmer, X_val_oh).numpy()

hybrid_metrics = evaluate_logits(val_logits, y_val)
print("hybrid student:", hybrid_metrics)
print("params:", count_params(hybrid))

# Mesurer la latence manuellement pour le modèle hybride
import time
hybrid.eval()
with torch.no_grad():
    start = time.time()
    for _ in range(100):
        _ = hybrid(X_val_kmer[:1], X_val_oh[:1])
    latency_ms = (time.time() - start) * 1000 / 100
print(f"latency (ms/sample): {latency_ms:.4f}")

hybrid student: {'accuracy': 0.852, 'f1': 0.8511066398390342}
params: 47780
latency (ms/sample): 0.2863


In [23]:
fusion_weights = torch.softmax(hybrid.fusion_weight, dim=0).detach().numpy()
print(f"Fusion weights: k-mer={fusion_weights[0]:.3f}, CNN={fusion_weights[1]:.3f}")

# Entraîner le même modèle hybride SANS distillation (alpha=1.0) pour comparer
hybrid_sup = HybridKmerCNN(kmer_dim=X_train_kmer.shape[1], seq_len=WINDOW, dropout=DROPOUT)
optimizer_sup = torch.optim.Adam(hybrid_sup.parameters(), lr=LR)
for epoch in range(N_EPOCHS):
    perm = torch.randperm(n)
    for start in range(0, n, batch_size):
        idx = perm[start:start + batch_size]
        optimizer_sup.zero_grad()
        student_logits = hybrid_sup(X_train_kmer[idx], X_train_oh[idx])
        loss = torch.nn.functional.binary_cross_entropy_with_logits(student_logits.squeeze(), y_train_t[idx])
        loss.backward()
        optimizer_sup.step()
hybrid_sup.eval()
with torch.no_grad():
    val_logits_sup = hybrid_sup(X_val_kmer, X_val_oh).numpy()
hybrid_sup_metrics = evaluate_logits(val_logits_sup, y_val)
print(f"hybrid sans distillation: accuracy={hybrid_sup_metrics['accuracy']:.3f}")

Fusion weights: k-mer=0.594, CNN=0.406
hybrid sans distillation: accuracy=0.847


<div style="height: 3px; margin: 2em 0 1.5em 0; background: linear-gradient(90deg, #0969da 0%, #0969da 55%, rgba(9,105,218,0.15) 100%);"></div>

#### **Comparaison avec les modèles de référence**

In [24]:
import pandas as pd

comparison = pd.DataFrame([
    {"model": "teacher (Evo2+MLP)", **teacher_metrics,
     "params": count_params(teacher),
     "latency_ms": measure_latency_torch(teacher, torch.tensor(X_val_emb[:1], dtype=torch.float32))},
    {"model": "hybrid kmer+CNN (sans distillation)", **hybrid_sup_metrics,
     "params": count_params(hybrid_sup),
     "latency_ms": latency_ms},
    {"model": "hybrid kmer+CNN (avec distillation)", **hybrid_metrics,
     "params": count_params(hybrid),
     "latency_ms": latency_ms},
])
comparison

,model,accuracy,f1,params,latency_ms
0,teacher (Evo2+MLP),0.959,0.959526,524545,0.044627
1,hybrid kmer+CNN (sans distillation),0.847,0.848064,47780,0.286341
2,hybrid kmer+CNN (avec distillation),0.852,0.851107,47780,0.286341


#### **Point de contrôle**

Le modèle hybride devrait :
- Avoir une précision d'environ 0.84 (meilleur que le student k-mer seul à ~0.84)
- Être beaucoup plus petit que le teacher (quelques milliers vs millions de paramètres)
- Être rapide à l'inférence (pas besoin d'Evo2)
- Montrer des poids de fusion équilibrés entre k-mers et CNN

La distillation aide principalement à guider la fusion des deux représentations,
permettant au modèle d'apprendre quand faire confiance à chaque branche.

Suite : `04_compression_analysis_and_wrapup.ipynb` — mettre tous les modèles sur un seul graphique.

<div style="margin-top: 3em;">
  <div style="height: 3px; background: linear-gradient(90deg, #0969da 0%, #0969da 55%, rgba(9,105,218,0.15) 100%);"></div>
  <div style="display: flex; align-items: center; gap: 0.9em; padding: 1.1em 1em; font-size: 0.9em; color: #57606a; background-color: #f2f6fd; border-radius: 0 0 6px 6px;">
    <svg width="34" height="34" viewBox="0 0 34 34" fill="none" style="flex: 0 0 auto;">
      <path d="M9 3c0 7 16 7 16 14S9 24 9 31" stroke="#0969da" stroke-width="2" stroke-linecap="round"/>
      <path d="M25 3c0 7-16 7-16 14s16 7 16 14" stroke="#0969da" stroke-width="2" stroke-linecap="round" opacity="0.45"/>
      <circle cx="17" cy="10" r="1.8" fill="#0969da"/>
      <circle cx="17" cy="24" r="1.8" fill="#0969da"/>
    </svg>
    <div style="flex: 1 1 auto;">
      <div style="color: #0969da; font-weight: 600; letter-spacing: 0.03em;">Fin du Lab 3c</div>
      <div>Prochaine &eacute;tape &rarr; <code>lab3/04_compression_analysis_and_wrapup.ipynb</code></div>
    </div>
    <div style="flex: 0 0 auto; text-align: right; border-right: 2px solid #0969da; padding-right: 0.9em;">
      <div style="font-weight: 600; color: #24292f;">AfricaBP &middot; bioAI Workshop</div>
      <div style="font-size: 0.85em;">AfricaBP 2026 &mdash; De l'ADN aux mod&egrave;les compress&eacute;s</div>
    </div>
  </div>
</div>